# Flux Utility Solutions - Full Deployment Notebook

**Purpose**: Deploy the complete Flux ecosystem from scratch using Snowflake Notebooks

This notebook executes all deployment scripts in sequence to create:
- Database infrastructure and warehouses
- Core grid tables (substations, transformers, meters, customers)
- AMI time-series tables and aggregations
- Cortex AI services (Semantic View, Search, Agent)
- PostgreSQL integration for low-latency operations
- SPCS compute pools and services

**Estimated Time**: 15-30 minutes depending on data scale

---

## Configuration

Set your deployment parameters:

In [ ]:
# Deployment Configuration
DATABASE = "FLUX_PROD"           # Target database name
WAREHOUSE = "FLUX_PROD_MEDIUM"   # Compute warehouse
ADMIN_ROLE = "FLUX_ADMIN_ROLE"   # Admin role
USER_ROLE = "FLUX_USER_ROLE"     # End-user role
POSTGRES_INSTANCE = "FLUX_OPERATIONS_POSTGRES"
COMPUTE_POOL = "FLUX_INTERACTIVE_POOL"

# Data scale: 'small' (quick demo) or 'full' (production scale)
DATA_SCALE = "small"

print(f"Deploying to: {DATABASE}")
print(f"Warehouse: {WAREHOUSE}")
print(f"Data Scale: {DATA_SCALE}")

In [ ]:
# Get Snowflake session
from snowflake.snowpark.context import get_active_session
session = get_active_session()

# Verify connection
result = session.sql("SELECT CURRENT_USER(), CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect()
print(f"User: {result[0][0]}")
print(f"Role: {result[0][1]}")
print(f"Warehouse: {result[0][2]}")

## Phase 1: Database Infrastructure

In [ ]:
# Create database and schemas
session.sql(f"CREATE DATABASE IF NOT EXISTS {DATABASE}").collect()
session.sql(f"USE DATABASE {DATABASE}").collect()

schemas = ['PRODUCTION', 'APPLICATIONS', 'RAW', 'ML']
for schema in schemas:
    session.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}").collect()
    print(f"✓ Created schema: {schema}")

print(f"\n✓ Database {DATABASE} ready with {len(schemas)} schemas")

In [ ]:
# Create warehouses
session.sql(f"""
    CREATE WAREHOUSE IF NOT EXISTS {WAREHOUSE}
    WAREHOUSE_SIZE = 'MEDIUM'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    COMMENT = 'Primary compute warehouse for Flux operations'
""").collect()

print(f"✓ Warehouse {WAREHOUSE} created")

## Phase 2: Core Grid Tables

In [ ]:
# Create SUBSTATIONS table
session.sql(f"USE SCHEMA {DATABASE}.PRODUCTION").collect()

session.sql("""
    CREATE TABLE IF NOT EXISTS SUBSTATIONS (
        SUBSTATION_ID VARCHAR(20) NOT NULL PRIMARY KEY,
        SUBSTATION_NAME VARCHAR(100),
        REGION VARCHAR(50),
        SUBSTATION_TYPE VARCHAR(20),
        VOLTAGE_LEVEL VARCHAR(10),
        LATITUDE FLOAT NOT NULL,
        LONGITUDE FLOAT NOT NULL,
        CAPACITY_MVA NUMBER(10,0),
        CURRENT_LOAD_MW NUMBER(10,2),
        PEAK_LOAD_MW NUMBER(10,2),
        OPERATIONAL_STATUS VARCHAR(20),
        COMMISSIONED_DATE DATE
    )
    COMMENT = 'Distribution and transmission substations'
""").collect()
print("✓ SUBSTATIONS table created")

In [ ]:
# Create TRANSFORMER_METADATA table
session.sql("""
    CREATE TABLE IF NOT EXISTS TRANSFORMER_METADATA (
        TRANSFORMER_ID VARCHAR(50) NOT NULL,
        LATITUDE FLOAT,
        LONGITUDE FLOAT,
        SUBSTATION_ID VARCHAR(50),
        CIRCUIT_ID VARCHAR(50),
        RATED_KVA NUMBER(10,0),
        INSTALL_YEAR NUMBER(4,0),
        HEALTH_SCORE FLOAT,
        CURRENT_LOAD_KVA NUMBER(20,1),
        PEAK_LOAD_KVA NUMBER(20,1),
        LOAD_UTILIZATION_PCT NUMBER(10,2),
        METER_COUNT NUMBER(10,0)
    )
    CLUSTER BY (SUBSTATION_ID, CIRCUIT_ID)
    COMMENT = 'Distribution transformers - 91,000 assets'
""").collect()
print("✓ TRANSFORMER_METADATA table created")

In [ ]:
# Create METER_INFRASTRUCTURE table
session.sql("""
    CREATE TABLE IF NOT EXISTS METER_INFRASTRUCTURE (
        METER_ID VARCHAR(50) NOT NULL,
        TRANSFORMER_ID VARCHAR(50),
        CIRCUIT_ID VARCHAR(50),
        SUBSTATION_ID VARCHAR(50),
        METER_LATITUDE FLOAT,
        METER_LONGITUDE FLOAT,
        CITY VARCHAR(100),
        ZIP_CODE VARCHAR(10),
        COUNTY_NAME VARCHAR(50),
        METER_TYPE VARCHAR(20),
        CUSTOMER_SEGMENT_ID VARCHAR(20),
        COMMISSIONED_DATE DATE,
        HEALTH_SCORE FLOAT
    )
    CLUSTER BY (TRANSFORMER_ID, CIRCUIT_ID)
    COMMENT = 'Smart meter infrastructure - 597,000 meters'
""").collect()
print("✓ METER_INFRASTRUCTURE table created")

In [ ]:
# Create CUSTOMERS_MASTER_DATA table
session.sql("""
    CREATE TABLE IF NOT EXISTS CUSTOMERS_MASTER_DATA (
        CUSTOMER_ID VARCHAR(50) NOT NULL,
        FIRST_NAME VARCHAR(50),
        LAST_NAME VARCHAR(50),
        FULL_NAME VARCHAR(100),
        PRIMARY_METER_ID VARCHAR(50),
        CUSTOMER_SEGMENT VARCHAR(20),
        SERVICE_ADDRESS VARCHAR(200),
        CITY VARCHAR(100),
        ZIP_CODE NUMBER(10,0),
        SERVICE_COUNTY VARCHAR(50),
        PHONE VARCHAR(20),
        EMAIL VARCHAR(100),
        ACCOUNT_STATUS VARCHAR(10),
        SERVICE_START_DATE DATE,
        CUSTOMER_EMBEDDING VECTOR(FLOAT, 768)
    )
    CLUSTER BY (CITY, ZIP_CODE, CUSTOMER_SEGMENT)
    COMMENT = 'Customer master data - 686,000 profiles'
""").collect()
print("✓ CUSTOMERS_MASTER_DATA table created")

## Phase 3: AMI Time-Series Tables

In [ ]:
# Create AMI_INTERVAL_READINGS table
session.sql("""
    CREATE TABLE IF NOT EXISTS AMI_INTERVAL_READINGS (
        METER_ID VARCHAR(50) NOT NULL,
        TIMESTAMP TIMESTAMP_NTZ NOT NULL,
        USAGE_KWH FLOAT,
        VOLTAGE NUMBER(10,0),
        POWER_FACTOR NUMBER(5,2),
        CUSTOMER_SEGMENT_ID VARCHAR(20),
        SOURCE_TABLE VARCHAR(50)
    )
    CLUSTER BY (DATE_TRUNC('DAY', TIMESTAMP), METER_ID)
    COMMENT = 'AMI interval readings - 7.1B rows, 15-minute granularity'
""").collect()
print("✓ AMI_INTERVAL_READINGS table created")

In [ ]:
# Create aggregation tables
session.sql("""
    CREATE TABLE IF NOT EXISTS TRANSFORMER_HOURLY_LOAD (
        TRANSFORMER_ID VARCHAR(50) NOT NULL,
        HOUR_START TIMESTAMP_NTZ NOT NULL,
        TOTAL_KWH NUMBER(15,2),
        AVG_VOLTAGE NUMBER(10,2),
        METER_COUNT NUMBER(10,0),
        PEAK_KWH NUMBER(10,2),
        LOAD_FACTOR_PCT NUMBER(10,2),
        PRIMARY KEY (TRANSFORMER_ID, HOUR_START)
    )
    CLUSTER BY (DATE_TRUNC('DAY', HOUR_START), TRANSFORMER_ID)
    COMMENT = 'Hourly transformer load aggregations - 211M rows'
""").collect()
print("✓ TRANSFORMER_HOURLY_LOAD table created")

## Phase 4: Verify Deployment

In [ ]:
# Verify all tables created
tables = session.sql(f"""
    SELECT table_name, row_count 
    FROM {DATABASE}.INFORMATION_SCHEMA.TABLES 
    WHERE table_schema = 'PRODUCTION'
    ORDER BY table_name
""").to_pandas()

print("DEPLOYMENT VERIFICATION")
print("=" * 50)
print(f"Database: {DATABASE}")
print(f"Tables created: {len(tables)}")
print("\nTable List:")
for _, row in tables.iterrows():
    print(f"  ✓ {row['TABLE_NAME']}: {row['ROW_COUNT'] or 0:,} rows")

## Next Steps

1. **Load Seed Data**: Use `generators/load_seed_data.py` or Flux Data Forge
2. **Create Cortex Services**: Run `08_semantic_view.sql`, `09_cortex_search_services.sql`
3. **Deploy SPCS Apps**: Deploy Flux Ops Center and Flux Data Forge
4. **Run Demo Notebooks**: Explore data with `ami_analytics.ipynb` and `customer_360_search.ipynb`